# Fairness in Machine Learning Lab

**Group members:** Da and Ma

## Project objective

This project audits and mitigates fairness disparities in a binary
credit-risk classifier trained on the German Credit dataset.

The sensitive attribute is age:

- **Young:** age < 25
- **Old:** age >= 25

The prediction target is:

- **1:** good credit
- **0:** bad credit

## Fairness families examined

1. **Independence:** demographic parity
2. **Separation:** equal opportunity and equalized odds
3. **Sufficiency:** predictive parity and calibration

## Normative principle

This notebook does not search for a universally fairest model.

For every metric and mitigation, we will ask:

1. Which definition of fairness does it optimise?
2. What happens to the other fairness definitions?
3. What predictive cost is incurred?
4. Which people bear that cost?

## Reproducibility rules

Throughout the project, we will use:

- one fixed random seed;
- one fixed train/test split;
- one consistent positive-class definition;
- one consistent age-group definition;
- the same test observations for comparable models;
- age retained separately for auditing, even when removed from the
  predictive features.

Generated tables will be saved in:

`outputs/tables/`

Generated figures will be saved in:

`outputs/figures/`

Saved models and preprocessing objects will be placed in:

`outputs/models/`

In [1]:
from __future__ import annotations

import json
import platform
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import sklearn

In [2]:
# Find the project root robustly.
#
# This works when JupyterLab is started from:
# 1. the fairness-lab project directory, or
# 2. the notebooks directory.

CURRENT_DIRECTORY = Path.cwd().resolve()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
elif (CURRENT_DIRECTORY / "src").exists():
    PROJECT_ROOT = CURRENT_DIRECTORY
else:
    raise RuntimeError(
        "Could not locate the project root. Start JupyterLab from the "
        "fairness-lab directory or from fairness-lab/notebooks."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Current working directory: {CURRENT_DIRECTORY}")
print(f"Project root: {PROJECT_ROOT}")

Current working directory: /home/davis/fairness-lab/notebooks
Project root: /home/davis/fairness-lab


In [3]:
from src.fairness_lab import (
    AGE_THRESHOLD,
    FIGURES_DIR,
    MODELS_DIR,
    NEGATIVE_LABEL,
    OLD_GROUP,
    OUTPUT_DIR,
    POSITIVE_LABEL,
    RANDOM_STATE,
    TABLES_DIR,
    TEST_SIZE,
    YOUNG_GROUP,
    define_age_group,
    initialise_project,
)

In [4]:
environment_info = initialise_project()

Fairness lab project initialised.
Project root: /home/davis/fairness-lab
Random seed: 42
Environment information: /home/davis/fairness-lab/outputs/environment.json


In [5]:
environment_info

{'python_version': '3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]',
 'python_implementation': 'CPython',
 'operating_system': 'Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39',
 'machine': 'x86_64',
 'random_state': 42,
 'test_size': 0.3,
 'positive_label': 1,
 'negative_label': 0,
 'age_threshold': 25,
 'young_group_definition': 'age < 25',
 'old_group_definition': 'age >= 25',
 'package_versions': {'numpy': '2.5.0',
  'pandas': '3.0.3',
  'matplotlib': '3.11.0',
  'scikit-learn': '1.9.0',
  'fairlearn': '0.14.0',
  'openml': '0.15.1',
  'jupyterlab': '4.6.0',
  'ipykernel': '7.3.0'}}

In [6]:
# Verify the central experimental configuration.

assert RANDOM_STATE == 42
assert TEST_SIZE == 0.30

assert POSITIVE_LABEL == 1
assert NEGATIVE_LABEL == 0

assert AGE_THRESHOLD == 25
assert YOUNG_GROUP == "young"
assert OLD_GROUP == "old"

assert define_age_group(18) == YOUNG_GROUP
assert define_age_group(24) == YOUNG_GROUP
assert define_age_group(25) == OLD_GROUP
assert define_age_group(40) == OLD_GROUP

print("All project configuration checks passed.")

All project configuration checks passed.


In [7]:
# Verify that the expected directories now exist.

required_directories = [
    OUTPUT_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    MODELS_DIR,
    PROJECT_ROOT / "report",
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "src",
]

for directory in required_directories:
    assert directory.exists(), f"Missing directory: {directory}"
    print(f"Found: {directory.relative_to(PROJECT_ROOT)}")

Found: outputs
Found: outputs/tables
Found: outputs/figures
Found: outputs/models
Found: report
Found: notebooks
Found: src


In [8]:
# Display the principal software versions used in the project.

software_versions = pd.DataFrame(
    {
        "Software": [
            "Python",
            "NumPy",
            "pandas",
            "Matplotlib",
            "scikit-learn",
        ],
        "Version": [
            platform.python_version(),
            np.__version__,
            pd.__version__,
            matplotlib.__version__,
            sklearn.__version__,
        ],
    }
)

software_versions

,Software,Version
0,Python,3.12.3
1,NumPy,2.5.0
2,pandas,3.0.3
3,Matplotlib,3.11.0
4,scikit-learn,1.9.0


# Pre-computation commitments

Before examining the dataset or running the model, Da and Ma recorded
their predictions and normative preferences in `00_commitments.md`.

That file represents the group's genuine pre-analysis position and
must not be edited after the experiments begin.

Any later reflections or changes of interpretation will be documented
separately.

In [9]:
commitments_path = PROJECT_ROOT / "00_commitments.md"

if not commitments_path.exists():
    raise FileNotFoundError(
        "00_commitments.md was not found.\n"
        "Da and Ma must complete and save their pre-computation "
        "commitments before beginning Mission 1."
    )

commitments_text = commitments_path.read_text(encoding="utf-8")

print(commitments_text)

In [10]:
# Record basic file information without changing the commitment file.

commitments_information = {
    "path": str(commitments_path),
    "size_bytes": commitments_path.stat().st_size,
    "modified_timestamp": commitments_path.stat().st_mtime,
}

commitments_information

{'path': '/home/davis/fairness-lab/00_commitments.md',
 'size_bytes': 0,
 'modified_timestamp': 1782484936.3596625}

# Mission 1 — The disparity

## Question

Is the baseline classifier unfair across the two age groups, and under
which fairness definition?

## Required outputs

For each group:

- number of observations;
- base rate of good credit;
- selection rate;
- true-positive rate;
- false-positive rate;
- false-negative rate;
- positive predictive value;
- confusion-matrix counts.

For the model overall:

- accuracy;
- demographic-parity difference;
- equalized-odds difference.

The complete audit will then be repeated after age is removed from the
predictive features while being retained separately for fairness
measurement.

In [11]:
# Mission 1 status

mission_1_status = {
    "mission": 1,
    "title": "The disparity",
    "implemented": False,
    "next_step": "Load and inspect the OpenML credit-g dataset.",
}

mission_1_status

{'mission': 1,
 'title': 'The disparity',
 'implemented': False,
 'next_step': 'Load and inspect the OpenML credit-g dataset.'}

# Mission 2 — The impossibility

## Question

Can the model have equal calibration and equal false-positive rates
across groups when their base rates differ?

For each group, we will verify:

\[
\mathrm{FPR}
=
\left(\frac{p}{1-p}\right)
\left(\frac{1-\mathrm{PPV}}{\mathrm{PPV}}\right)
(1-\mathrm{FNR})
\]

where:

- \(p\) is the base rate of the positive class;
- PPV is positive predictive value;
- FNR is false-negative rate;
- FPR is false-positive rate.

The identity will be calculated from the model's own confusion-matrix
counts.

In [12]:
# Mission 2 status

mission_2_status = {
    "mission": 2,
    "title": "The impossibility",
    "implemented": False,
    "next_step": (
        "Use the Mission 1 group confusion matrices to verify "
        "Chouldechova's identity."
    ),
}

mission_2_status

{'mission': 2,
 'title': 'The impossibility',
 'implemented': False,
 'next_step': "Use the Mission 1 group confusion matrices to verify Chouldechova's identity."}

# Mission 3 — Mitigation and fairness–accuracy frontiers

The following approaches will be compared:

1. Baseline classifier
2. Reweighing
3. ExponentiatedGradient
4. ThresholdOptimizer

For each operating point, we will report:

- accuracy;
- demographic-parity difference;
- equalized-odds difference;
- group-specific TPR and FPR.

Two separate frontier figures will be produced:

1. accuracy versus demographic-parity difference;
2. accuracy versus equalized-odds difference.

The two fairness measures will not be combined into a generic
single fairness score.

In [13]:
# Mission 3 status

mission_3_status = {
    "mission": 3,
    "title": "Mitigation and fairness–accuracy frontiers",
    "implemented": False,
    "methods": [
        "Baseline",
        "Reweighing",
        "ExponentiatedGradient",
        "ThresholdOptimizer",
    ],
}

mission_3_status

{'mission': 3,
 'title': 'Mitigation and fairness–accuracy frontiers',
 'implemented': False,
 'methods': ['Baseline',
  'Reweighing',
  'ExponentiatedGradient',
  'ThresholdOptimizer']}

# Mission 4 — Chosen operating point

Da and Ma will choose exactly one operating point from the experimental
results.

The choice will not be made automatically by selecting:

- the highest accuracy;
- the smallest demographic-parity difference;
- the smallest equalized-odds difference;
- a generic combined fairness score.

The selected operating point must be defended as a normative decision.

The final argument must identify:

1. the fairness criterion being prioritised;
2. the people protected by that criterion;
3. the predictive or operational cost;
4. who bears that cost;
5. the limitations of the historical target label.

In [14]:
# Mission 4 status

mission_4_status = {
    "mission": 4,
    "title": "Chosen operating point",
    "implemented": False,
    "decision_makers": ["Da", "Ma"],
    "automatic_selection_allowed": False,
}

mission_4_status

{'mission': 4,
 'title': 'Chosen operating point',
 'implemented': False,
 'decision_makers': ['Da', 'Ma'],
 'automatic_selection_allowed': False}

# Mission 5 — Regulation and deployment limits

The chosen credit-scoring system will be examined as a potentially
deployed product.

The legal analysis will address:

- high-risk classification under the EU AI Act;
- data and bias governance;
- use of sensitive information for bias detection;
- human oversight;
- automated decision-making safeguards;
- disparate treatment and disparate impact.

The final section will identify at least:

1. two concrete legal or regulatory obligations;
2. one concrete human-oversight mechanism;
3. a reasoned deployment conclusion.

In [15]:
# Mission 5 status

mission_5_status = {
    "mission": 5,
    "title": "Regulation and deployment limits",
    "implemented": False,
    "primary_sources_required": True,
    "minimum_obligations": 2,
    "minimum_oversight_measures": 1,
}

mission_5_status

{'mission': 5,
 'title': 'Regulation and deployment limits',
 'implemented': False,
 'primary_sources_required': True,
 'minimum_obligations': 2,
 'minimum_oversight_measures': 1}

# Project status summary

This section confirms that the Step 2 project foundation is operational.

At this stage:

- the environment has been recorded;
- the random seed has been fixed;
- the output directories have been created;
- the sensitive-group definition has been tested;
- the positive-class definition has been tested;
- the pre-computation commitments have been loaded;
- the five mission sections have been prepared.

The experimental implementation begins with Mission 1.

In [16]:
project_status = pd.DataFrame(
    [
        {
            "Component": "Project directories",
            "Status": "Ready",
        },
        {
            "Component": "Environment metadata",
            "Status": "Ready",
        },
        {
            "Component": "Random seed",
            "Status": f"Fixed at {RANDOM_STATE}",
        },
        {
            "Component": "Positive label",
            "Status": f"Good credit = {POSITIVE_LABEL}",
        },
        {
            "Component": "Young group",
            "Status": f"Age < {AGE_THRESHOLD}",
        },
        {
            "Component": "Old group",
            "Status": f"Age >= {AGE_THRESHOLD}",
        },
        {
            "Component": "Commitment file",
            "Status": "Loaded",
        },
        {
            "Component": "Mission 1 analysis",
            "Status": "Not yet implemented",
        },
        {
            "Component": "Mission 2 analysis",
            "Status": "Not yet implemented",
        },
        {
            "Component": "Mission 3 analysis",
            "Status": "Not yet implemented",
        },
        {
            "Component": "Mission 4 decision",
            "Status": "Pending experimental results",
        },
        {
            "Component": "Mission 5 analysis",
            "Status": "Not yet implemented",
        },
    ]
)

project_status

,Component,Status
0,Project directories,Ready
1,Environment metadata,Ready
2,Random seed,Fixed at 42
3,Positive label,Good credit = 1
4,Young group,Age < 25
5,Old group,Age >= 25
6,Commitment file,Loaded
7,Mission 1 analysis,Not yet implemented
8,Mission 2 analysis,Not yet implemented
9,Mission 3 analysis,Not yet implemented


In [17]:
# Final Step 2 checks

assert commitments_path.exists()
assert OUTPUT_DIR.exists()
assert TABLES_DIR.exists()
assert FIGURES_DIR.exists()
assert MODELS_DIR.exists()

environment_file = OUTPUT_DIR / "environment.json"
assert environment_file.exists()

with environment_file.open("r", encoding="utf-8") as file:
    saved_environment = json.load(file)

assert saved_environment["random_state"] == RANDOM_STATE
assert saved_environment["positive_label"] == POSITIVE_LABEL
assert saved_environment["negative_label"] == NEGATIVE_LABEL
assert saved_environment["age_threshold"] == AGE_THRESHOLD

print("Step 2 completed successfully.")
print("The project is ready for Mission 1.")

Step 2 completed successfully.
The project is ready for Mission 1.


# Mission 1 — Baseline disparity audit

In this mission, we will:

1. Load the German Credit dataset from OpenML.
2. Encode the target as:
   - good credit = 1
   - bad credit = 0
3. Define the sensitive age group:
   - young = age < 25
   - old = age >= 25
4. Train a baseline classifier.
5. Audit the classifier by age group.
6. Remove age from the predictive features.
7. Retrain and re-audit the model.

Important distinction:

- Age may be used as a predictive feature in the baseline model.
- Age must remain available separately for auditing.
- Removing age from features does not mean removing age from fairness measurement.

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from fairlearn.metrics import (
    MetricFrame,
    count,
    demographic_parity_difference,
    equalized_odds_difference,
    false_negative_rate,
    false_positive_rate,
    selection_rate,
    true_positive_rate,
)

In [19]:
# Load OpenML German Credit dataset.
# OpenML name: credit-g

credit_data = fetch_openml(
    name="credit-g",
    version=1,
    as_frame=True,
    parser="auto",
)

raw_X = credit_data.data.copy()
raw_y = credit_data.target.copy()

print(f"Dataset name: {credit_data.details.get('name')}")
print(f"Dataset version: {credit_data.details.get('version')}")
print(f"Rows: {raw_X.shape[0]}")
print(f"Feature columns: {raw_X.shape[1]}")
print("\nTarget values:")
print(raw_y.value_counts())

Dataset name: credit-g
Dataset version: 1
Rows: 1000
Feature columns: 20

Target values:
class
good    700
bad     300
Name: count, dtype: int64


In [20]:
# Basic dataset inspection.

display(raw_X.head())

dataset_overview = pd.DataFrame(
    {
        "column": raw_X.columns,
        "dtype": [str(raw_X[col].dtype) for col in raw_X.columns],
        "missing_values": [raw_X[col].isna().sum() for col in raw_X.columns],
        "unique_values": [raw_X[col].nunique(dropna=True) for col in raw_X.columns],
    }
)

dataset_overview

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,4,real estate,67,none,own,2,skilled,1,yes,yes
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,2,real estate,22,none,own,1,skilled,1,none,yes
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,3,real estate,49,none,own,1,unskilled resident,2,none,yes
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,4,life insurance,45,none,for free,1,skilled,2,none,yes
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,4,no known property,53,none,for free,2,skilled,2,none,yes


,column,dtype,missing_values,unique_values
0,checking_status,category,0,4
1,duration,int64,0,33
2,credit_history,category,0,5
3,purpose,category,0,10
4,credit_amount,int64,0,921
5,savings_status,category,0,5
6,employment,category,0,5
7,installment_commitment,int64,0,4
8,personal_status,category,0,4
9,other_parties,category,0,3


In [21]:
# Encode the target.
#
# Positive class:
# good credit = 1
#
# Negative class:
# bad credit = 0

target_mapping = {
    "good": POSITIVE_LABEL,
    "bad": NEGATIVE_LABEL,
}

y = raw_y.map(target_mapping).astype(int)

print(y.value_counts().sort_index())
print(f"Positive-class rate: {y.mean():.3f}")

class
0    300
1    700
Name: count, dtype: int64
Positive-class rate: 0.700


In [22]:
# Define the sensitive attribute from age.

X = raw_X.copy()

if "age" not in X.columns:
    raise KeyError("Expected an 'age' column in the German Credit dataset.")

X["age"] = pd.to_numeric(X["age"], errors="raise")

sensitive = X["age"].apply(define_age_group)

sensitive_counts = sensitive.value_counts().rename_axis("age_group").reset_index(name="count")
sensitive_counts

,age_group,count
0,old,851
1,young,149


In [23]:
# Save the basic sensitive-group counts.

sensitive_counts_path = TABLES_DIR / "mission1_sensitive_group_counts.csv"
sensitive_counts.to_csv(sensitive_counts_path, index=False)

print(f"Saved: {sensitive_counts_path.relative_to(PROJECT_ROOT)}")

Saved: outputs/tables/mission1_sensitive_group_counts.csv


In [24]:
# Create one fixed train/test split.
#
# The same split must be reused for the baseline model and the age-removed model.

X_train, X_test, y_train, y_test, sensitive_train, sensitive_test = train_test_split(
    X,
    y,
    sensitive,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_indices = pd.DataFrame(
    {
        "index": list(X_train.index) + list(X_test.index),
        "split": ["train"] * len(X_train) + ["test"] * len(X_test),
    }
)

split_indices_path = TABLES_DIR / "mission1_train_test_indices.csv"
split_indices.to_csv(split_indices_path, index=False)

print(f"Train rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Saved: {split_indices_path.relative_to(PROJECT_ROOT)}")

Train rows: 700
Test rows: 300
Saved: outputs/tables/mission1_train_test_indices.csv


In [25]:
import sys
from pathlib import Path

possible_roots = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path.home() / "fairness-lab",
]

PROJECT_ROOT = None

for candidate in possible_roots:
    if (candidate / "src" / "mission1_utils.py").exists():
        PROJECT_ROOT = candidate.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find fairness-lab/src/mission1_utils.py. "
        "Check that the notebook is inside ~/fairness-lab/notebooks "
        "or that the project exists at ~/fairness-lab."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root set to: {PROJECT_ROOT}")
print(f"First sys.path entry: {sys.path[0]}")

Project root set to: /home/davis/fairness-lab
First sys.path entry: /home/davis/fairness-lab


In [26]:
from src.mission1_utils import audit_model, build_logistic_regression_model, confusion_counts_by_group

In [27]:
print("Mission 1 helper functions imported successfully.")

Mission 1 helper functions imported successfully.


In [28]:
baseline_model = build_logistic_regression_model(X_train)
baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

baseline_group_metrics, baseline_summary = audit_model(
    model_name="Baseline with age",
    y_true=y_test,
    y_pred=baseline_predictions,
    sensitive_features=sensitive_test,
)

display(baseline_group_metrics)
display(baseline_summary)

,model,group,n,base_rate,selection_rate,TPR,FPR,FNR,PPV,TN,FP,FN,TP
0,Baseline with age,old,253.0,0.727273,0.770751,0.869565,0.507246,0.130435,0.820513,34,35,24,160
1,Baseline with age,young,47.0,0.553191,0.659574,0.730769,0.571429,0.269231,0.612903,9,12,7,19


,model,accuracy,selection_rate,DP_difference,EO_difference
0,Baseline with age,0.74,0.753333,0.111177,0.138796


In [29]:
# Save Mission 1 baseline outputs.

baseline_group_metrics_path = TABLES_DIR / "mission1_baseline_group_metrics.csv"
baseline_summary_path = TABLES_DIR / "mission1_baseline_summary.csv"

baseline_group_metrics.to_csv(baseline_group_metrics_path, index=False)
baseline_summary.to_csv(baseline_summary_path, index=False)

print(f"Saved: {baseline_group_metrics_path.relative_to(PROJECT_ROOT)}")
print(f"Saved: {baseline_summary_path.relative_to(PROJECT_ROOT)}")

Saved: outputs/tables/mission1_baseline_group_metrics.csv
Saved: outputs/tables/mission1_baseline_summary.csv


## Mission 1 age-removal test

Now we remove `age` from the predictive features and retrain the model.

We do **not** remove age from the sensitive-feature vector, because we
still need it to audit disparities.

This tests the claim:

> The model never sees the protected attribute, so it cannot discriminate.

In [30]:
# Remove age from the predictive features only.

X_without_age = X.drop(columns=["age"])

X_train_no_age = X_without_age.loc[X_train.index].copy()
X_test_no_age = X_without_age.loc[X_test.index].copy()

print(f"Original feature count: {X_train.shape[1]}")
print(f"Feature count without age: {X_train_no_age.shape[1]}")

Original feature count: 20
Feature count without age: 19


In [31]:
# Train and audit the model without age.

no_age_model = build_logistic_regression_model(X_train_no_age)
no_age_model.fit(X_train_no_age, y_train)

no_age_predictions = no_age_model.predict(X_test_no_age)

no_age_group_metrics, no_age_summary = audit_model(
    model_name="Model without age",
    y_true=y_test,
    y_pred=no_age_predictions,
    sensitive_features=sensitive_test,
)

display(no_age_group_metrics)
display(no_age_summary)

,model,group,n,base_rate,selection_rate,TPR,FPR,FNR,PPV,TN,FP,FN,TP
0,Model without age,old,253.0,0.727273,0.770751,0.875000,0.492754,0.125000,0.825641,35,34,23,161
1,Model without age,young,47.0,0.553191,0.659574,0.730769,0.571429,0.269231,0.612903,9,12,7,19


,model,accuracy,selection_rate,DP_difference,EO_difference
0,Model without age,0.746667,0.753333,0.111177,0.144231


In [32]:
# Save age-removed outputs.

no_age_group_metrics_path = TABLES_DIR / "mission1_age_removed_group_metrics.csv"
no_age_summary_path = TABLES_DIR / "mission1_age_removed_summary.csv"

no_age_group_metrics.to_csv(no_age_group_metrics_path, index=False)
no_age_summary.to_csv(no_age_summary_path, index=False)

print(f"Saved: {no_age_group_metrics_path.relative_to(PROJECT_ROOT)}")
print(f"Saved: {no_age_summary_path.relative_to(PROJECT_ROOT)}")

Saved: outputs/tables/mission1_age_removed_group_metrics.csv
Saved: outputs/tables/mission1_age_removed_summary.csv


In [33]:
# Compare baseline and age-removed models.

mission1_model_comparison = pd.concat(
    [baseline_summary, no_age_summary],
    ignore_index=True,
)

mission1_model_comparison_path = TABLES_DIR / "mission1_model_comparison.csv"
mission1_model_comparison.to_csv(mission1_model_comparison_path, index=False)

mission1_model_comparison

,model,accuracy,selection_rate,DP_difference,EO_difference
0,Baseline with age,0.740000,0.753333,0.111177,0.138796
1,Model without age,0.746667,0.753333,0.111177,0.144231


In [34]:
# Prepare a short interpretation aid.
#
# These are not the final report sentences. Da and Ma must still interpret
# the results carefully.

def compare_group_metric(group_metrics: pd.DataFrame, metric_name: str) -> pd.DataFrame:
    return (
        group_metrics[["model", "group", metric_name]]
        .pivot(index="model", columns="group", values=metric_name)
        .reset_index()
    )


for metric_name in ["base_rate", "selection_rate", "TPR", "FPR", "FNR", "PPV"]:
    print(f"\n{metric_name}")
    display(compare_group_metric(
        pd.concat([baseline_group_metrics, no_age_group_metrics], ignore_index=True),
        metric_name,
    ))


base_rate


group,model,old,young
0,Baseline with age,0.727273,0.553191
1,Model without age,0.727273,0.553191



selection_rate


group,model,old,young
0,Baseline with age,0.770751,0.659574
1,Model without age,0.770751,0.659574



TPR


...

group,model,old,young
0,Baseline with age,0.507246,0.571429
1,Model without age,0.492754,0.571429



FNR


group,model,old,young
0,Baseline with age,0.130435,0.269231
1,Model without age,0.125000,0.269231



PPV


group,model,old,young
0,Baseline with age,0.820513,0.612903
1,Model without age,0.825641,0.612903


## Mission 1 interpretation questions

Da and Ma must now answer these questions using the tables above:

1. Which group has the lower base rate of good credit?
2. Which group has the lower selection rate?
3. Which group has the lower TPR?
4. Which group has the higher FPR?
5. Which group has the lower PPV?
6. Is the same group disadvantaged under every metric?
7. Did removing age make the demographic-parity difference zero?
8. Did removing age make the equalized-odds difference zero?
9. Which variables may plausibly act as age proxies?
10. What is your verdict on “fairness through unawareness”?